Test Spacy's POS tagging for extracting actions from recipes.

In [1]:
import spacy

!spacy info en_core_web_trf --url

https://github.com/explosion/spacy-models/releases/download/en_core_web_trf-3.8.0/en_core_web_trf-3.8.0-py3-none-any.whl


In [3]:
nlp = spacy.load("en_core_web_trf")
doc = nlp("She has been writing a report and will submit it tomorrow.")

verbs = [
    {"text": token.text, "lemma": token.lemma_, "pos": token.pos_}
    for token in doc
    if token.pos_ in {"VERB"}
]

print(verbs)

[{'text': 'writing', 'lemma': 'write', 'pos': 'VERB'}, {'text': 'submit', 'lemma': 'submit', 'pos': 'VERB'}]


Test `en_core_web_trf` on the Recipes dataset.

First, we select the first 5 direction samples from our dataset and cast to a list.

In [7]:
import polars as pl

df = pl.read_csv("../data/RecipeNLG_dataset.csv")

sample_directions = df.select("directions").head(5).to_series().to_list()

for sample in sample_directions:
    print(sample)

["In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine.", "Stir over medium heat until mixture bubbles all over top.", "Boil and stir 5 minutes more. Take off heat.", "Stir in vanilla and cereal; mix well.", "Using 2 teaspoons, drop and shape into 30 clusters on wax paper.", "Let stand until firm, about 30 minutes."]
["Place chipped beef on bottom of baking dish.", "Place chicken on top of beef.", "Mix soup and cream together; pour over chicken. Bake, uncovered, at 275\u00b0 for 3 hours."]
["In a slow cooker, combine all ingredients. Cover and cook on low for 4 hours or until heated through and cheese is melted. Stir well before serving. Yields 6 servings."]
["Boil and debone chicken.", "Put bite size pieces in average size square casserole dish.", "Pour gravy and cream of mushroom soup over chicken; level.", "Make stuffing according to instructions on box (do not make too moist).", "Put stuffing on top of chicken and gravy; level.", "Sprinkle shr

Next, define the function to extract the verbs.

In [25]:
import json

def extract_verbs_from_text(directions_list):
    directions_list = json.loads(directions_list)
    verbs = []
    for text in directions_list:
        doc = nlp(text)
        for token in doc:
            if token.pos_ == "VERB":
                verbs.append(token.text)
    return str(verbs)

Apply the function.

In [27]:
for sample in sample_directions:
    print(sample)
    verbs = extract_verbs_from_text(sample)
    print(f"Sample: {sample}")
    print(f"Extracted Verbs: {verbs}")
    print(f"Type: {type(verbs)}")
    print("-" * 40)

["In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine.", "Stir over medium heat until mixture bubbles all over top.", "Boil and stir 5 minutes more. Take off heat.", "Stir in vanilla and cereal; mix well.", "Using 2 teaspoons, drop and shape into 30 clusters on wax paper.", "Let stand until firm, about 30 minutes."]


Sample: ["In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine.", "Stir over medium heat until mixture bubbles all over top.", "Boil and stir 5 minutes more. Take off heat.", "Stir in vanilla and cereal; mix well.", "Using 2 teaspoons, drop and shape into 30 clusters on wax paper.", "Let stand until firm, about 30 minutes."]
Extracted Verbs: ['mix', 'evaporated', 'Stir', 'bubbles', 'Boil', 'stir', 'Take', 'Stir', 'mix', 'Using', 'drop', 'shape', 'stand']
Type: <class 'str'>
----------------------------------------
["Place chipped beef on bottom of baking dish.", "Place chicken on top of beef.", "Mix soup and cream together; pour over chicken. Bake, uncovered, at 275\u00b0 for 3 hours."]
Sample: ["Place chipped beef on bottom of baking dish.", "Place chicken on top of beef.", "Mix soup and cream together; pour over chicken. Bake, uncovered, at 275\u00b0 for 3 hours."]
Extracted Verbs: ['Place', 'chipped', 'Place', 'Mix', 'pour', 'Bake', 'uncovered'